In [ ]:
# Import package
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import igraph
import scvelo as scv
import loompy as lmp
import anndata as ad
import harmonypy as hm
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

import celltypist
from celltypist import models
from celltypist.models import Model

# scANVI label transfer 결과 확인

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq.h5ad")
Liver_sn

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='dataset_id',
    frameon=False,
    title='Journal dataset',
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='predictions_scanvi',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='Disease_status',
    frameon=False,
    title='Disease state',
    size=10
)

In [ ]:
Liver_sn.obs['Disease_status'].unique()

In [ ]:
# 원하는 순서 정의
desired_order = ["Healthy", "MASL", "MASLD", "MASH", "Cirrhosis", "end stage" ]
Liver_sn.obs['Disease_status'] = pd.Categorical(Liver_sn.obs['Disease_status'], categories=desired_order, ordered=True)

cluster_labels = Liver_sn.obs['predictions_scanvi']
batch_labels = Liver_sn.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

# 3. matplotlib barplot 그릴 때 palette 지정
crosstab.plot(
    kind='bar',
    stacked=True,
    figsize=(5, 10),
    #color=[palette_dict[col] for col in crosstab.columns]
)

plt.title('Major cell lineages in MASLD stages')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='Disease state cellannot', bbox_to_anchor=(1.05, 1), loc='upper left')

In [ ]:
Liver_sn.obs

In [ ]:
# Hepatocyte annot
sc.pl.dotplot(Liver_sn,['FGB','GHR','TENM2','CYP3A4','CPS1','ASS1'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# Cholangiocyte annot
sc.pl.dotplot(Liver_sn,['CFTR','BICC1','PKHD1','CTNND2','RALYL','KRT19','KRT7','SPP1','MUC1','MUC6'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# Stellete annot
sc.pl.dotplot(Liver_sn,['DCN','COL3A1','PTH1R','ADAMTSL2','CCBE1','ADAMTS2','ADAMTSL1'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# Endothelial annot
sc.pl.dotplot(Liver_sn,['KDR','FCN3','FLT1','OIT3','STAB2','TEK','PTPRB','FCN2'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# Lymphocytes annot
sc.pl.dotplot(Liver_sn,['IL7R','CD96','BCL11B','THEMIS','CD247','PTPRC'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# Macrophage & Neutrophil annot
sc.pl.dotplot(Liver_sn,['NDST3','MARCO','CD163','MSR1','GAS7','CD5L','FCN1','ITGAX'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
# B cells annot
sc.pl.dotplot(Liver_sn,['BANK1','IGKC','PAX5','FCRL5'], groupby="predictions_scanvi", standard_scale='var')

In [ ]:
marker_genes = {
    'Hepatocyte':['FGB','GHR','TENM2','CYP3A4','CPS1','ASS1'],
    'Cholangiocyte': ['CFTR','BICC1','PKHD1','CTNND2','RALYL','KRT19','KRT7','SPP1','MUC1','MUC6'],
    'Stellete': ['DCN','COL3A1','PTH1R','ADAMTSL2','CCBE1','ADAMTS2','ADAMTSL1'],
    'Endothelial': ['KDR','FCN3','FLT1','OIT3','STAB2','TEK','PTPRB','FCN2'],
    'Lymphocyte': ['IL7R','CD96','BCL11B','THEMIS','CD247','PTPRC'],
    'Myeloid': ['NDST3','MARCO','CD163','MSR1','GAS7','CD5L','FCN1','ITGAX'],
    'B cells': ['BANK1','IGKC','PAX5','FCRL5']
}

In [ ]:
# 순서 정의
stage_order = ["Hepatocyte", "Cholangiocyte", "Stellete", "Endothelial", "Lymphocyte", "Macrophage", "B cells"]
Liver_sn.obs['predictions_scanvi'] = pd.Categorical(Liver_sn.obs['predictions_scanvi'], categories=stage_order, ordered=True)

In [ ]:
# B cells annot
sc.pl.dotplot(Liver_sn,marker_genes, groupby="predictions_scanvi", standard_scale='var')

In [ ]:
Liver_sn.obs

In [ ]:
Liver_sn.obs['fibrosis'] = (
    Liver_sn.obs['fibrosis']
    .astype(str)               # 문자열 변환
    .replace(['nan', 'NaN', 'None', 'Nan', 'NAN'], np.nan)
)


obs_nonan = Liver_sn.obs.dropna(subset=['fibrosis']).copy()
obs_nonan['score'] = obs_nonan['fibrosis'].astype(int)
crosstab_score = pd.crosstab(
    obs_nonan['fibrosis'],
    obs_nonan['predictions_scanvi'],
    normalize='index' 
)

crosstab_score.plot(
    kind='bar',
    stacked=True,
    figsize=(6, 5)
)

plt.title('Cluster composition by fibrosis score (excluding NaN)')
plt.xlabel('fibrosis Score')
plt.ylabel('Proportion of cells')
plt.legend(title='Cell type (predictions_scanvi)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
crosstab_prop = pd.crosstab(
    obs_nonan['fibrosis'],
    obs_nonan['predictions_scanvi'],
    normalize='index'
)

crosstab_count = pd.crosstab(
    obs_nonan['fibrosis'],
    obs_nonan['predictions_scanvi']
)

counts_per_stage = crosstab_count.sum(axis=1)

ax = crosstab_prop.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 9)
)

plt.title('Cluster composition by fibrosis score (excluding NaN)')
plt.xlabel('Fibrosis Score')
plt.ylabel('Proportion of cells')
plt.legend(
    title='Cell type (predictions_scanvi)',
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

for idx, p in enumerate(ax.patches):
    # 각 fibrosis 그룹마다 한 번만 표시하기 위해 마지막 bar 위에서 표시
    if (idx + 1) % len(crosstab_prop.columns) == 0:
        stage = crosstab_prop.index[idx // len(crosstab_prop.columns)]
        total_cells = counts_per_stage.loc[stage]
        x = p.get_x() + p.get_width() / 2
        y = 0.5
        ax.text(x, y, f"n={total_cells:,}", ha='center', va='bottom', fontsize=9, rotation=90)

plt.tight_layout()
plt.show()

In [ ]:
adata_GSE202379=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE202379_Raw.h5ad")
adata_GSE189600=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE189600_Raw.h5ad")
adata_GSE210077=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE210077_Raw.h5ad")
adata_GSE212837=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE212837_Raw.h5ad")
adata_inhouse=sc.read("/home/jaehyunchoi/Liver_bio_project/Nucseq/Data/Process_data/Nuc_raw.h5ad")

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq.h5ad")
Liver_sn

In [ ]:
Liver_sn.obs

# Total Liver sn merged analysis _ preprocessing

In [ ]:
adata_inhouse.var["mt"] = adata_inhouse.var_names.str.startswith("MT-")
adata_inhouse.var["ribo"] = adata_inhouse.var_names.str.startswith(("RPS", "RPL"))
adata_inhouse.var["hb"] = adata_inhouse.var_names.str.contains("^HB[^(P)(E)]")
sc.pp.calculate_qc_metrics(adata_inhouse, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
sc.pp.filter_genes(adata_inhouse, min_cells=3, inplace=True)
adata_inhouse = adata_inhouse[
    (adata_inhouse.obs['total_counts'] >= 800) &           # UMI >= 800
    (adata_inhouse.obs['pct_counts_mt'] < 10) &            # 미토콘드리아 유전자 비율 < 10%
    (adata_inhouse.obs['pct_counts_ribo'] < 10) &          # 리보솜 유전자 비율 < 10%
    (adata_inhouse.obs['total_counts'] < 50000) &          # 너무 큰 outlier 제거
    (adata_inhouse.obs['n_genes_by_counts'] > 1000)        # 최소 발현 유전자 수 > 1000
]
adata_inhouse

In [ ]:
adata_inhouse.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/inhouse_Raw.h5ad")

In [ ]:
adata_GSE202379=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE202379_Raw.h5ad")
adata_GSE189600=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE189600_Raw.h5ad")
adata_GSE210077=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE210077_Raw.h5ad")
adata_GSE212837=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/GSE212837_Raw.h5ad")
adata_inhouse=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/inhouse_Raw.h5ad")

In [ ]:
adatas = [adata_GSE202379,adata_GSE210077,adata_GSE212837, adata_GSE189600, adata_inhouse]

In [ ]:
adata_merged = sc.concat(
    adatas,
    join='inner',
    label='journal',
    keys=['GSE202379','GSE212837','GSE210077','GSE18960','inHouse'],
    index_unique=None
)

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq.h5ad")
Liver_sn

In [ ]:
Liver_sn

In [ ]:
adata_merged 

In [ ]:
adata_merged.obs

In [ ]:
Liver_sn.obs

In [ ]:
def rename_barcodes_with_sample(adata):
    """
    기존 바코드에서 마지막 '-' 뒤의 숫자 제거 후 sample 이름을 붙임.
    예: AACAGGGAGCGCTTCG-33 -> AACAGGGAGCGCTTCG-GSM6112221
    """
    if 'sample' not in adata.obs.columns:
        raise ValueError("⚠️ adata.obs에 'sample' 컬럼이 없습니다!")
    base_barcodes = adata.obs_names.str.replace(r'-\d+$', '', regex=True)
    new_barcodes = base_barcodes + '-' + adata.obs['sample'].astype(str)
    if not new_barcodes.is_unique:
        raise ValueError("⚠️ 새로운 바코드가 중복됩니다. sample 이름 또는 기존 바코드 확인 필요!")
    adata.obs_names = new_barcodes
    print(f"✅ {adata.shape[0]:,}개 셀의 바코드를 sample 기반으로 업데이트 완료.")
    return adata

In [ ]:
Liver_sn_raw = rename_barcodes_with_sample(adata_merged)
Liver_cellannot = rename_barcodes_with_sample(Liver_sn)

In [ ]:
# 두 AnnData의 셀 이름 set 비교
set1 = set(Liver_sn_raw.obs_names)
set2 = set(Liver_cellannot.obs_names)

# 기본 정보 출력
print(f"adata1 셀 개수: {len(set1):,}")
print(f"adata2 셀 개수: {len(set2):,}")
print(f"공통된 셀 개수: {len(set1 & set2):,}")
print(f"adata1에만 있는 셀: {len(set1 - set2):,}")
print(f"adata2에만 있는 셀: {len(set2 - set1):,}")
print(f"일치율: {len(set1 & set2) / len(set1) * 100:.2f}%")

# 완전 동일 여부 (순서까지)
print("순서까지 완전히 동일?:", Liver_sn_raw.obs_names.equals(Liver_cellannot.obs_names))

In [ ]:
import pandas as pd

# 1️⃣ 각 adata의 (barcode, sample) 조합 DataFrame 생성
df1 = pd.DataFrame({
    'barcode': Liver_sn_raw.obs_names,
    'sample': Liver_sn_raw.obs['sample'].astype(str).values
})

df2 = pd.DataFrame({
    'barcode': Liver_cellannot.obs_names,
    'sample': Liver_cellannot.obs['sample'].astype(str).values
})

# 2️⃣ 두 조합을 문자열로 병합 (유일 식별자)
df1['barcode_sample'] = df1['barcode'] + '-' + df1['sample']
df2['barcode_sample'] = df2['barcode'] + '-' + df2['sample']

# 3️⃣ set 비교
set1 = set(df1['barcode_sample'])
set2 = set(df2['barcode_sample'])

# 4️⃣ 일치율 계산
intersection = len(set1 & set2)
union = len(set1 | set2)
match_rate = intersection / len(set1) * 100

print(f"🔬 총 셀 수: {len(set1):,}")
print(f"✅ 일치한 barcode-sample 조합: {intersection:,}")
print(f"❌ 다른 barcode-sample 조합: {len(set1 - set2):,}")
print(f"🎯 일치율: {match_rate:.4f}%")

# 5️⃣ 순서까지 완전히 동일한지 확인
pairwise_equal = (df1['barcode_sample'].values == df2['barcode_sample'].values).all()
print("순서 + 조합까지 완전히 동일?:", pairwise_equal)

In [ ]:
Liver_sn_raw.obs['celltype'] = Liver_cellannot.obs['predictions_scanvi'].reindex(Liver_sn_raw.obs_names).values


print("✅ Liver_sn_raw.obs['celltype'] 복사 완료")
print("값 개요:\n", Liver_sn_raw.obs['celltype'].value_counts().head())
print("NaN 비율:", Liver_sn_raw.obs['celltype'].isna().mean())

In [ ]:
Liver_sn_raw.obs['gender'] = Liver_cellannot.obs['Sex'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['Disease_status'] = Liver_cellannot.obs['Disease_status'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['age'] = Liver_cellannot.obs['Age'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['steatosis'] = Liver_cellannot.obs['steatosis'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['activity'] = Liver_cellannot.obs['activity'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['fibrosis'] = Liver_cellannot.obs['fibrosis'].reindex(Liver_sn_raw.obs_names).values
Liver_sn_raw.obs['NAS5'] = Liver_cellannot.obs['NAS5'].reindex(Liver_sn_raw.obs_names).values

In [ ]:
import numpy as np

for key in ['X_scANVI', 'X_scVI', 'X_umap']:
    if key in Liver_cellannot.obsm.keys():
        print(f"Processing {key} ...")
        
        # Liver_cellannot의 좌표를 DataFrame으로 변환 (index = cell barcode)
        coords = pd.DataFrame(
            Liver_cellannot.obsm[key],
            index=Liver_cellannot.obs_names
        )
        
        # Liver_sn_raw 순서에 맞게 reindex (바코드 기준 align)
        coords_aligned = coords.reindex(Liver_sn_raw.obs_names)
        
        # 확인: reindex 후 누락된 셀 없는지
        if coords_aligned.isna().any().any():
            missing = coords_aligned.isna().any(axis=1).sum()
            print(f"⚠️ Warning: {missing} cells missing alignment in {key}")
        
        # numpy array로 복원하여 복사
        Liver_sn_raw.obsm[key] = coords_aligned.to_numpy()
        print(f"✅ {key} copied and aligned by cell barcode.")
    else:
        print(f"⚠️ {key} not found in Liver_cellannot.obsm")

In [ ]:
sc.pl.umap(
    Liver_cellannot,
    color='predictions_scanvi',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn_raw,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
Liver_sn_raw

In [ ]:
Liver_cellannot

In [ ]:
Liver_sn_raw.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")

# Liver DB nucseq 01

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")
Liver_sn_raw

In [ ]:
Liver_sn_raw.raw = Liver_sn_raw
sc.pp.normalize_total(Liver_sn_raw, target_sum=1e4)
sc.pp.log1p(Liver_sn_raw)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(Liver_sn_raw, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(Liver_sn_raw, max_value=10)
# Run PCA
sc.tl.pca(Liver_sn_raw, svd_solver='arpack')
print(Liver_sn_raw.obsm['X_pca'][:5, :5])
sc.pl.pca(Liver_sn_raw, color='sample')

import matplotlib.pyplot as plt
var_ratio = Liver_sn_raw.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
sc.pl.embedding(Liver_sn_raw, basis='X_scANVI', color='journal', dimensions=[(0, 1)])

In [ ]:
sc.pl.umap(
    Liver_sn_raw,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
Liver_sn_raw.obs

In [ ]:
# 원하는 순서 정의
desired_order = ["Healthy", "MASL", "MASLD", "MASH", "Cirrhosis", "end stage" ]
Liver_sn_raw.obs['Disease_status'] = pd.Categorical(Liver_sn_raw.obs['Disease_status'], categories=desired_order, ordered=True)

cluster_labels = Liver_sn_raw.obs['celltype']
batch_labels = Liver_sn_raw.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

crosstab.plot(kind='bar', stacked=True, figsize=(5, 7))
plt.title('Major cell lineages in MASLD stage')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='MASLD stage', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
sc.pl.umap(
    Liver_sn_raw,
    color='SLC7A5',
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn_raw,
    color='CYP3A4',
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn_raw,
    color='CD3D',
    frameon=False,
    size=10
)

In [ ]:
Liver_sn_raw.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")

In [ ]:
Liver_sn_raw

In [ ]:
Liver_stage = Liver_sn_raw[Liver_sn_raw.obs['Disease_status'] != 'MASLD'].copy()

In [ ]:
Liver_stage.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")

In [ ]:
Liver_stage

In [ ]:
Liver_stage.obs['Disease_status'].value_counts()

In [ ]:
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
Liver_stage.obs['stage_numeric'].value_counts()

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    ['EIF2AK4','ATF4','SESN2','MIOS','SEC13','SEH1L','WDR24','WDR59','MTOR'],
    groupby="stage",
    standard_scale=None
)

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    ['SIRT1','PRKAA1','PRKAA2','EIF2AK4'], # SIRT1, AMPK (PRKAA1, PRKAA2), GCN2
    groupby="stage",
    standard_scale=None
)

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    ['SIRT1','PRKAA1','PRKAA2','EIF2AK4'], # SIRT1, AMPK (PRKAA1, PRKAA2), GCN2
    groupby="stage",
    standard_scale='var'
)

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    ['EIF2AK4','ATF4','SESN2','MIOS','SEC13','SEH1L','WDR24','WDR59','MTOR'],
    groupby="stage",
    standard_scale='var'
)

In [ ]:
genes = ['EIF2AK4','ATF4','SESN2','MIOS','SEC13','SEH1L','WDR24','WDR59','MTOR']
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.5 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        Liver_stage,
        keys=gene,
        groupby='stage',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
plt.show()

In [ ]:
# 원하는 순서 정의
desired_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage" ]
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=desired_order, ordered=True)

cluster_labels = Liver_stage.obs['celltype']
batch_labels = Liver_stage.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

crosstab.plot(kind='bar', stacked=True, figsize=(5, 7))
plt.title('Major cell lineages in MASLD stage')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

In [ ]:
genes = ['CRP', 'ITIH4', 'DCDC2', 'CYP3A7', 'BICC1', 'SAA2', 'DTNA']
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.5 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        Liver_stage,
        keys=gene,
        groupby='stage',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
plt.show()

In [ ]:
genes = ['FKBP5', 'LINC-PINT', 'GADD45G', 'CTNNA3', 'FOSB']
n_genes = len(genes)
fig, axs = plt.subplots(n_genes, 1, figsize=(6, 1.5 * n_genes))  # 세로 크기를 줄임

# axs가 하나일 경우 (genes가 1개) 대비
if n_genes == 1:
    axs = [axs]

for i, gene in enumerate(genes):
    sc.pl.violin(
        Liver_stage,
        keys=gene,
        groupby='stage',
        ax=axs[i],
        show=False,
        stripplot=False,
        jitter=False
    )
    axs[i].set_title(gene)

plt.tight_layout()
plt.show()

In [ ]:
Zonal_Markers = {
    'periportal': ["HAL","MFSD2A","IGFBP2","PCK1","ASS1","SDS"],
    'pericentral': ["CYP3A4","SLC1A2","SLCO1B3","AXIN2","LGR5","GLUL"]}

In [ ]:
Markers = {
    'Zone1': ['CPS1', 'HAL', 'ARG1', 'SLC38A2', 'SLC1A5'],
    'Zone3': ['GLUL', 'CYP2E1', 'CYP1A2', 'CD36', 'SLC7A5', 'SLC16A10'],
    'Bile':  ['ABCB11', 'ABCC2', 'ALB', 'KRT19']
}

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    Zonal_Markers,
    groupby="stage",
    standard_scale= None
)

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    Zonal_Markers,
    groupby="stage",
    standard_scale= "var"
)

In [ ]:
sc.pl.matrixplot(
    Liver_stage,
    Zonal_Markers,
    groupby="stage",
    standard_scale= "var"
)

In [ ]:
sc.pl.matrixplot(
    Liver_stage,
    Markers,
    groupby="stage",
    standard_scale= "var"
)

# AGE correlation gene analysis

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import scanpy as sc

In [ ]:
Liver_stage=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")

In [ ]:
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
Liver_stage.obs

In [ ]:
adata_normal = Liver_stage[(Liver_stage.obs['stage'] == 'Healthy')].copy()
print(f"세포 수: {adata_normal.n_obs:,}, 유전자 수: {adata_normal.n_vars:,}")
print("stage 값:", adata_normal.obs['stage'].unique().tolist())

In [ ]:
# expr 정의 추가
expr = adata_normal.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse matrix인 경우 dense로 변환

In [ ]:
# ages 변수 준비 (stages 대신)
ages = adata_normal.obs['age'].values  # 'age' 컬럼명은 실제 데이터에 맞게 수정

corrs = []
pvals = []
for i in range(expr.shape[1]):
    r, p = spearmanr(ages, expr[:, i])  # stages → ages
    corrs.append(r)
    pvals.append(p)

df = pd.DataFrame({
    'gene': adata_normal.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

# age 증가에 따라 발현 증가하는 유전자만 선택
df_pos = df[(df['spearman_r'] > 0) & (df['pval'] < 0.05)]
df_pos = df_pos.sort_values('spearman_r', ascending=False)
df_pos.head(10)

df_neg = df[(df['spearman_r'] < 0) & (df['pval'] < 0.05)]
df_neg = df_neg.sort_values('spearman_r', ascending=False)
df_neg.head(10)

df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_Healthy_age_correlation.csv")

In [ ]:
adata_masl = Liver_stage[(Liver_stage.obs['stage'] == 'MASL')].copy()
print(f"세포 수: {adata_masl.n_obs:,}, 유전자 수: {adata_masl.n_vars:,}")
print("stage 값:", adata_masl.obs['stage'].unique().tolist())

# expr 정의 추가
expr = adata_masl.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse matrix인 경우 dense로 변환
    
# ages 변수 준비 (stages 대신)
ages = adata_masl.obs['age'].values  # 'age' 컬럼명은 실제 데이터에 맞게 수정

corrs = []
pvals = []
for i in range(expr.shape[1]):
    r, p = spearmanr(ages, expr[:, i])  # stages → ages
    corrs.append(r)
    pvals.append(p)

df = pd.DataFrame({
    'gene': adata_masl.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

# age 증가에 따라 발현 증가하는 유전자만 선택
df_pos = df[(df['spearman_r'] > 0) & (df['pval'] < 0.05)]
df_pos = df_pos.sort_values('spearman_r', ascending=False)
df_pos.head(10)

df_neg = df[(df['spearman_r'] < 0) & (df['pval'] < 0.05)]
df_neg = df_neg.sort_values('spearman_r', ascending=False)
df_neg.head(10)

df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_MASL_age_correlation.csv")

In [ ]:
adata_mash = Liver_stage[(Liver_stage.obs['stage'] == 'MASH')].copy()
print(f"세포 수: {adata_mash.n_obs:,}, 유전자 수: {adata_mash.n_vars:,}")
print("stage 값:", adata_mash.obs['stage'].unique().tolist())

# expr 정의 추가
expr = adata_mash.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse matrix인 경우 dense로 변환
    
# ages 변수 준비 (stages 대신)
ages = adata_mash.obs['age'].values  # 'age' 컬럼명은 실제 데이터에 맞게 수정

corrs = []
pvals = []
for i in range(expr.shape[1]):
    r, p = spearmanr(ages, expr[:, i])  # stages → ages
    corrs.append(r)
    pvals.append(p)

df = pd.DataFrame({
    'gene': adata_mash.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

# age 증가에 따라 발현 증가하는 유전자만 선택
df_pos = df[(df['spearman_r'] > 0) & (df['pval'] < 0.05)]
df_pos = df_pos.sort_values('spearman_r', ascending=False)
df_pos.head(10)

df_neg = df[(df['spearman_r'] < 0) & (df['pval'] < 0.05)]
df_neg = df_neg.sort_values('spearman_r', ascending=False)
df_neg.head(10)

df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_MASH_age_correlation.csv")

# MASLD stage correlation gene analysis

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import scanpy as sc

In [ ]:
Liver_stage=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")

In [ ]:
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
stages = Liver_stage.obs['stage_numeric'].values
expr = Liver_stage.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df = pd.DataFrame({
    'gene': Liver_stage.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

# stage 증가에 따라 발현 증가하는 유전자만 선택
df_pos = df[(df['spearman_r'] > 0) & (df['pval'] < 0.05)]
df_pos = df_pos.sort_values('spearman_r', ascending=False)
df_pos.head(10)
df_neg = df[(df['spearman_r'] < 0) & (df['pval'] < 0.05)]
df_neg = df_neg.sort_values('spearman_r', ascending=False)
df_neg.head(10)

df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_correlation.csv")

In [ ]:
df

In [ ]:
adata_hepa = Liver_stage[(Liver_stage.obs['celltype'] == 'Hepatocyte')].copy()
print(f"세포 수: {adata_hepa.n_obs:,}, 유전자 수: {adata_hepa.n_vars:,}")
print("stage 값:", adata_hepa.obs['stage'].unique().tolist())

In [ ]:
stages = adata_hepa.obs['stage_numeric'].values
expr = adata_hepa.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_hep = pd.DataFrame({
    'gene': adata_hepa.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
# stage 증가에 따라 발현 증가하는 유전자만 선택
df_pos = df_hep[(df_hep['spearman_r'] > 0) & (df_hep['pval'] < 0.05)]
df_pos = df_pos.sort_values('spearman_r', ascending=False)
df_pos.head(10)


In [ ]:
df_neg = df_hep[(df_hep['spearman_r'] < 0) & (df_hep['pval'] < 0.05)]
df_neg = df_neg.sort_values('spearman_r', ascending=False)
df_neg.head(10)

In [ ]:
df_hep.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_hepa_correlation.csv")

In [ ]:
gene = "NEAT1" 
sc.pl.violin(
    adata_hepa,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1)

In [ ]:
gene = "CYP3A4" 
sc.pl.violin(
    adata_hepa,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1)

In [ ]:
adata_endo = Liver_stage[(Liver_stage.obs['celltype'] == 'Endothelial')].copy()
print(f"세포 수: {adata_endo.n_obs:,}, 유전자 수: {adata_endo.n_vars:,}")
print("stage 값:", adata_endo.obs['stage'].unique().tolist())
stages = adata_endo.obs['stage_numeric'].values
expr = adata_endo.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_endo = pd.DataFrame({
    'gene': adata_endo.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_endo

In [ ]:
df_endo.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_endo_correlation.csv")

In [ ]:
adata_stelle = Liver_stage[(Liver_stage.obs['celltype'] == 'Stellete')].copy()
print(f"세포 수: {adata_stelle.n_obs:,}, 유전자 수: {adata_endo.n_vars:,}")
print("stage 값:", adata_stelle.obs['stage'].unique().tolist())
stages = adata_stelle.obs['stage_numeric'].values
expr = adata_stelle.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_stelle = pd.DataFrame({
    'gene': adata_stelle.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_stelle

In [ ]:
df_stelle.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_stellete_correlation.csv")

In [ ]:
adata_chol = Liver_stage[(Liver_stage.obs['celltype'] == 'Cholangiocyte')].copy()
print(f"세포 수: {adata_chol.n_obs:,}, 유전자 수: {adata_endo.n_vars:,}")
print("stage 값:", adata_chol.obs['stage'].unique().tolist())
stages = adata_chol.obs['stage_numeric'].values
expr = adata_chol.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_chol = pd.DataFrame({
    'gene': adata_chol.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_chol

In [ ]:
df_chol.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_cholangiocyte_correlation.csv")

In [ ]:
adata_mac = Liver_stage[(Liver_stage.obs['celltype'] == 'Macrophage')].copy()
print(f"세포 수: {adata_mac.n_obs:,}, 유전자 수: {adata_mac.n_vars:,}")
print("stage 값:", adata_mac.obs['stage'].unique().tolist())
stages = adata_mac.obs['stage_numeric'].values
expr = adata_mac.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_mac = pd.DataFrame({
    'gene': adata_mac.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_mac

In [ ]:
df_mac.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_macrophage_correlation.csv")

In [ ]:
adata_lym = Liver_stage[(Liver_stage.obs['celltype'] == 'Lymphocyte')].copy()
print(f"세포 수: {adata_lym.n_obs:,}, 유전자 수: {adata_mac.n_vars:,}")
print("stage 값:", adata_lym.obs['stage'].unique().tolist())
stages = adata_lym.obs['stage_numeric'].values
expr = adata_lym.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_lym = pd.DataFrame({
    'gene': adata_lym.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_lym

In [ ]:
df_lym.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_lymphoid_correlation.csv")

In [ ]:
adata_B = Liver_stage[(Liver_stage.obs['celltype'] == 'B cells')].copy()
print(f"세포 수: {adata_B.n_obs:,}, 유전자 수: {adata_B.n_vars:,}")
print("stage 값:", adata_B.obs['stage'].unique().tolist())
stages = adata_B.obs['stage_numeric'].values
expr = adata_B.X
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()  # sparse -> dense
print("expression matrix shape:", expr.shape)

assert len(stages) == expr.shape[0], "⚠️ stage 길이와 expression 행 수가 다릅니다!"

In [ ]:
corrs = []
pvals = []

for i in range(expr.shape[1]):
    r, p = spearmanr(stages, expr[:, i])
    corrs.append(r)
    pvals.append(p)

df_B = pd.DataFrame({
    'gene': adata_B.var_names,
    'spearman_r': corrs,
    'pval': pvals
})

In [ ]:
df_B

In [ ]:
df_B.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_B_correlation.csv")

In [ ]:
Liver_stage

In [ ]:
pd.crosstab(Liver_stage.obs['journal'], Liver_stage.obs['celltype'])

# GSE202379 pericentral- periportal correlation

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")
Liver_sn

In [ ]:
adata_hepa = Liver_sn[(Liver_sn.obs['celltype'] == 'Hepatocyte')].copy()
adata_hepa

In [ ]:
adata_hepa.obs

In [ ]:
pericentral_genes = ["CYP2E1", "CYP1A2", "GLUL", "HAL", "ARG1"]
periportal_genes  = ["CPS1", "HMGCS2", "PCK1", "ASS1", "ALDOB"]
genes = pericentral_genes + periportal_genes

In [ ]:
genes = ["HAL","MFSD2A","IGFBP2","PCK1","ASS1","SDS","CYP3A4","SLC1A2","SLCO1B3","AXIN2","LGR5","GLUL"]

In [ ]:
periportal_genes = ["HAL","MFSD2A","IGFBP2","PCK1","ASS1","SDS"]
pericentral_genes  = ["CYP3A4","SLC1A2","SLCO1B3","AXIN2","LGR5","GLUL"]
genes = pericentral_genes + periportal_genes

In [ ]:
genes_in_data = [g for g in genes if g in adata_hepa.var_names]
print(f"✅ 사용할 유전자 개수: {len(genes_in_data)} / {len(genes)}")

In [ ]:
stage_order = ["Healthy", "MASL", "MASLD", "MASH", "Cirrhosis", "end stage"]
adata_hepa.obs["Disease_status"] = pd.Categorical(adata_hepa.obs["Disease_status"], categories=stage_order, ordered=True)

In [ ]:
# file: plot_gene_correlation_pdf.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

genes = ["HAL","MFSD2A","IGFBP2","PCK1","ASS1","SDS","CYP3A4","SLC1A2","SLCO1B3","AXIN2","LGR5","GLUL"]
genes_in_data = [g for g in genes if g in adata_hepa.var_names]
print(f"✅ 사용할 유전자 개수: {len(genes_in_data)} / {len(genes)}")

In [ ]:
adata_hepa_healthy = adata_hepa[adata_hepa.obs["Disease_status"] == "Healthy"].copy()
expr = adata_hepa_healthy[:, genes_in_data].X

if not isinstance(expr, np.ndarray):
    expr = expr.toarray()

corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(corr_matrix, index=genes_in_data, columns=genes_in_data)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

# y축: 위→아래 HAL ... GLUL
# x축: 왼→오 HAL ... GLUL 이므로, 이를 뒤집어서
# 실제 표시를 왼쪽부터 GLUL ... HAL, 즉 오른쪽 끝이 HAL 이 되게 만듦
corr_plot = corr_no_diag.loc[genes_in_data, genes_in_data[::-1]]

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)

# 회색 대각선 추가
# x축은 reversed, y축은 original 이므로
# 같은 유전자 위치를 찾아서 해당 칸만 회색 처리
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )

ax.set_title("Healthy-Hepatocyte")
plt.tight_layout()

output_pdf = "/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Figure/Pericentral_Periportal_Healthy_Hepatocyte_correlation.pdf"
fig.savefig(output_pdf, format="pdf", bbox_inches="tight")
plt.show()

print(f"✅ PDF 저장 완료: {output_pdf}")

In [ ]:
adata_hepa_healthy = adata_hepa[adata_hepa.obs["Disease_status"] == "Healthy"].copy()
expr = adata_hepa_healthy[:, genes_in_data].X

if not isinstance(expr, np.ndarray):
    expr = expr.toarray()

corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(corr_matrix, index=genes_in_data, columns=genes_in_data)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

# y축: 위→아래 HAL ... GLUL
# x축: 왼→오 HAL ... GLUL 이므로, 이를 뒤집어서
# 실제 표시를 왼쪽부터 GLUL ... HAL, 즉 오른쪽 끝이 HAL 이 되게 만듦
corr_plot = corr_no_diag.loc[genes_in_data, genes_in_data[::-1]]

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)

# 회색 대각선 추가
# x축은 reversed, y축은 original 이므로
# 같은 유전자 위치를 찾아서 해당 칸만 회색 처리
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("Healthy-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
# MASL stage
adata_hepa_MASL = adata_hepa[(adata_hepa.obs['Disease_status'] == 'MASL')].copy()
expr = adata_hepa_MASL[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_MASL = corr_df.copy()
np.fill_diagonal(corr_MASL.values, np.nan)

# MASH stage
adata_hepa_MASH = adata_hepa[(adata_hepa.obs['Disease_status'] == 'MASH')].copy()
expr = adata_hepa_MASH[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_MASH = corr_df.copy()
np.fill_diagonal(corr_MASH.values, np.nan)

# Cirrhosis stage
adata_hepa_Cirrhosis = adata_hepa[(adata_hepa.obs['Disease_status'] == 'Cirrhosis')].copy()
expr = adata_hepa_Cirrhosis[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_Cirrhosis = corr_df.copy()
np.fill_diagonal(corr_Cirrhosis.values, np.nan)

# end stage
adata_hepa_End = adata_hepa[(adata_hepa.obs['Disease_status'] == 'end stage')].copy()
expr = adata_hepa_End[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_End = corr_df.copy()
np.fill_diagonal(corr_End.values, np.nan)

In [ ]:
corr_plot = corr_no_diag.loc[genes_in_data, genes_in_data[::-1]]
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag_r",
    center=0,
    vmin=-0.4,
    vmax=0.4,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)

# 회색 대각선 추가
# x축은 reversed, y축은 original 이므로
# 같은 유전자 위치를 찾아서 해당 칸만 회색 처리
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("Healthy-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
corr_plot = corr_MASL.loc[genes_in_data, genes_in_data[::-1]]
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag_r",
    center=0,
    vmin=-0.4,
    vmax=0.4,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)

x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("MASL-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
corr_plot = corr_MASH.loc[genes_in_data, genes_in_data[::-1]]
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag_r",
    center=0,
    vmin=-0.4,
    vmax=0.4,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("MASH-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
corr_plot = corr_Cirrhosis.loc[genes_in_data, genes_in_data[::-1]]
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag_r",
    center=0,
    vmin=-0.4,
    vmax=0.4,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("Cirrhosis-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
corr_plot = corr_End.loc[genes_in_data, genes_in_data[::-1]]
fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(
    corr_plot,
    cmap="vlag_r",
    center=0,
    vmin=-0.4,
    vmax=0.4,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax
)
x_order = list(corr_plot.columns)
y_order = list(corr_plot.index)

for gene in genes_in_data:
    i = y_order.index(gene)
    j = x_order.index(gene)
    ax.add_patch(
        plt.Rectangle((j, i), 1, 1, fill=True, color="gray", ec="gray", lw=0)
    )


ax.set_title("End stage-Hepatocyte")
plt.tight_layout()
plt.show()

In [ ]:
# Healthy stage
adata_hepa_healthy = adata_hepa[(adata_hepa.obs['Disease_status'] == 'Healthy')].copy()
expr = adata_hepa_healthy[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="vlag",
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("Pericentral–Periportal Gene Expression Healthy-Hepatocyte")
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="RdBu",
    center=0,
    square=True,
    #linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("Healthy")
plt.show()

In [ ]:
# Healthy stage
adata_hepa_MASL = adata_hepa[(adata_hepa.obs['Disease_status'] == 'MASL')].copy()
expr = adata_hepa_MASL[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="RdBu",
    center=0,
    square=True,
    #linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("MASL")
plt.show()

In [ ]:
# Healthy stage
adata_hepa_MASH = adata_hepa[(adata_hepa.obs['Disease_status'] == 'MASH')].copy()
expr = adata_hepa_MASH[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="RdBu",
    center=0,
    square=True,
    #linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("MASH")
plt.show()

In [ ]:
# Healthy stage
adata_hepa_Cirrhosis = adata_hepa[(adata_hepa.obs['Disease_status'] == 'Cirrhosis')].copy()
expr = adata_hepa_Cirrhosis[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="RdBu",
    center=0,
    square=True,
    #linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("Cirrhosis")
plt.show()

In [ ]:
# Healthy stage
adata_hepa_end = adata_hepa[(adata_hepa.obs['Disease_status'] == 'end stage')].copy()
expr = adata_hepa_end[:, genes_in_data].X

# sparse matrix일 경우 toarray()
if not isinstance(expr, np.ndarray):
    expr = expr.toarray()
corr_matrix = np.corrcoef(expr.T)
corr_df = pd.DataFrame(
    corr_matrix,
    index=genes_in_data,
    columns=genes_in_data
)

corr_no_diag = corr_df.copy()
np.fill_diagonal(corr_no_diag.values, np.nan)

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_no_diag,
    cmap="RdBu",
    center=0,
    square=True,
    #linewidths=0.5,
    cbar_kws={'label': 'Pearson correlation'}
)
for i in range(len(corr_gray_diag)):
    plt.gca().add_patch(
        plt.Rectangle((i, i), 1, 1, fill=True, color='gray', ec='gray', lw=0)
    )
plt.title("End stage")
plt.show()

In [ ]:
adata_hepa.obs["Disease_status"].value_counts()

# Nuc_DB DEG analysis

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")

In [ ]:
Liver_sn_raw

In [ ]:
Liver_sn_raw.obs['Disease_status'].unique()

In [ ]:
Liver_sn_raw.obs['Disease_MASLD'] = Liver_sn_raw.obs['Disease_status']

In [ ]:
Liver_sn_raw.obs['Disease_MASLD'] = Liver_sn_raw.obs['Disease_MASLD'].replace({
    'MASL': 'Disease',
    'MASLD': 'Disease',
    'MASH': 'Disease',
    'Healthy': 'Healthy'  # 그대로 유지
})

In [ ]:
Liver_sn_raw.obs['Disease_MASLD'].unique()

In [ ]:
sc.tl.rank_genes_groups(
    Liver_sn_raw,
    groupby="Disease_MASLD",
    groups=["Disease"],         # 분석할 그룹
    reference="Healthy",     # 비교할 기준 그룹
    method="wilcoxon",       # 또는 "t-test", "logreg"
    n_genes=Liver_sn_raw.shape[1]   # 전체 유전자 사용
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(Liver_sn_raw, group='Disease')
deg_df.head()

In [ ]:
deg_df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_HealthyvsMASL_MASH.csv")

In [ ]:
Liver_sn

In [ ]:
sc.tl.rank_genes_groups(
    Liver_sn,
    groupby="Disease_status",
    groups=["MASL"],         # 분석할 그룹
    reference="Healthy",     # 비교할 기준 그룹
    method="wilcoxon",       # 또는 "t-test", "logreg"
    n_genes=Liver_sn.shape[1]   # 전체 유전자 사용
)
deg_df = sc.get.rank_genes_groups_df(Liver_sn, group='MASL')
deg_df.head()

In [ ]:
deg_df

In [ ]:
deg_df.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_HealthyvsMASL.csv")

In [ ]:
sc.tl.rank_genes_groups(
    Liver_sn,
    groupby="Disease_status",
    groups=["MASH"],         # 분석할 그룹
    reference="MASL",     # 비교할 기준 그룹
    method="wilcoxon",       # 또는 "t-test", "logreg"
    n_genes=Liver_sn.shape[1]   # 전체 유전자 사용
)
deg_MASH = sc.get.rank_genes_groups_df(Liver_sn, group='MASH')
deg_MASH.head()

In [ ]:
deg_MASH

In [ ]:
deg_MASH.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_MASLvsMASH.csv")

In [ ]:
sc.tl.rank_genes_groups(
    Liver_sn,
    groupby="Disease_status",
    groups=["Cirrhosis"],         # 분석할 그룹
    reference="MASH",     # 비교할 기준 그룹
    method="wilcoxon",       # 또는 "t-test", "logreg"
    n_genes=Liver_sn.shape[1]   # 전체 유전자 사용
)
deg_cirr = sc.get.rank_genes_groups_df(Liver_sn, group='Cirrhosis')
deg_cirr.head()

In [ ]:
deg_cirr

In [ ]:
Liver_sn.obs["Disease_status"].value_counts()

In [ ]:
deg_cirr.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_MASHvsCirrho.csv")

In [ ]:
sc.tl.rank_genes_groups(
    Liver_sn,
    groupby="Disease_status",
    groups=["end stage"],         # 분석할 그룹
    reference="Cirrhosis",     # 비교할 기준 그룹
    method="wilcoxon",       # 또는 "t-test", "logreg"
    n_genes=Liver_sn.shape[1]   # 전체 유전자 사용
)
deg_end = sc.get.rank_genes_groups_df(Liver_sn, group='end stage')
deg_end.head()

In [ ]:
deg_end

In [ ]:
deg_end.to_csv("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Nuc_DB_total_CirrhovsEnd.csv")

In [ ]:
sc.pl.umap(
    Liver_sn,
    color=['GLUL','LGR5','AXIN2'],
    frameon=False,
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn,
    color=['HAL','MFSD2A','IGFBP2'],
    frameon=False,
    size=10
)

In [ ]:
import os
import tempfile

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import seaborn as sns
import torch

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")

In [ ]:
Liver_sn

In [ ]:
Liver_sn.obs['celltype']

In [ ]:
desired_order = ["Hepatocyte", "Cholangiocyte", "Endothelial", "Stellete", "Lymphocyte", "Macrophage","B cells" ]
Liver_sn.obs['celltype'] = pd.Categorical(Liver_sn.obs['celltype'], categories=desired_order, ordered=True)

In [ ]:
Markers = {
    'Hepatocyte': ['FGB', 'GHR', 'TENM2', 'CYP3A4', 'CPS1', 'ASS1'],
    'Cholangiocyte': ['CFTR', 'BICC1', 'PKHD1', 'RALYL', 'KRT19', 'KRT7','SPP1','MUC1','MUC6'],
    'Endothelial':  ['KDR', 'FCN3', 'FLT1', 'OIT3','STAB2','TEK','PTPRB','FCN2'],
    'Stellete':  ['DCN', 'COL3A1', 'PTH1R', 'ADAMTSL2','CCBE1','ADAMTS2','ADAMTSL1'],
    'Lymphocyte':  ['IL7R', 'CD96', 'BCL11B', 'THEMIS', 'CD247','PTPRC'],
    'Macrophage':  ['NDST3', 'MARCO', 'CD163', 'MSR1','GAS7','CD5L','FCN1','ITGAX'],
    'B_cells':  ['BANK1', 'IGKC', 'PAX5', 'FCRL5']
}

In [ ]:
sc.pl.dotplot(
    Liver_sn,
    Markers,
    groupby="celltype",
    standard_scale= None
)

In [ ]:
sc.pl.dotplot(
    Liver_sn,
    Markers,
    groupby="celltype",
    standard_scale= 'var'
)

In [ ]:
sc.pl.matrixplot(
    Liver_sn,
    Markers,
    groupby="celltype",
    standard_scale= None
)

In [ ]:
sc.pl.heatmap(
    Liver_sn,
    var_names=Markers,
    groupby="celltype",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 9)
)

In [ ]:
Liver_stage = Liver_sn[Liver_sn.obs['Disease_status'] != 'MASLD'].copy()
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
Transporters = {
    'Glucogenic': ['SLC6A19', 'SLC1A5', 'SLC7A8', 'SLC1A4', 'SLC38A1', 'SLC38A2'],
    'Ketogenic': ['SLC7A5', 'SLC16A10', 'SLC43A1', 'SLC3A2','SLC7A6']
}

In [ ]:
Liver_stage.obs['stage']

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    Transporters,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
sc.pl.matrixplot(
    Liver_stage,
    Transporters,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
sc.pl.heatmap(
    Liver_stage,
    var_names=Transporters,
    groupby="stage",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 9)
)

In [ ]:
genes = ['SLC1A1','SLC1A2','SLC1A4','SLC1A5','SLC7A2','SLC7A5','SLC7A6','SLC7A7','SLC7A8','SLC7A9','SLC7A11',
         'SLC16A10','SLC25A12', 'SLC25A13','SLC25A18','SLC25A22','SLC25A44','SLC36A1','SLC36A4','SLC38A1',
         'SLC38A2','SLC38A3','SLC38A4','SLC38A6','SLC38A7','SLC38A9','SLC38A10','SLC43A1','SLC43A2',]

In [ ]:
sc.pl.dotplot(
    Liver_stage,
    genes,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
sc.pl.matrixplot(
    Liver_stage,
    genes,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
sc.pl.heatmap(
    Liver_stage,
    var_names=genes,
    groupby="stage",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 9)
)

In [ ]:
Liver_stage.obs

In [ ]:
sc.pl.heatmap(
    Liver_stage,
    var_names=genes,
    groupby="celltype",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 9)
)

In [ ]:
adata_hepa = Liver_stage[(Liver_stage.obs['celltype'] == 'Hepatocyte')].copy()
adata_hepa

In [ ]:
sc.pl.matrixplot(
    adata_hepa,
    Transporters,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
sc.pl.matrixplot(
    adata_hepa,
    genes,
    groupby="stage",
    standard_scale= 'var'
)

In [ ]:
glucogen = ['AKT3','SIK','CREB3','ADCY2','PPARGC1A','CPT1C','CPT1A','CPT1B','CREB1','ATF2','CREBBP','PPP4R3C','CREB3L4','SIK1','LDHAL6A','CALML6','CRTC2','EP300','AKT1','AKT2','FBP1','FOXO1','SIK2','PLCB1','SIRT1','G6PC1','GCG','GCGR','GCK','GNAQ','GNAS','GYS1','GYS2','ACACA','ACACB','PRMT1','ITPR1','ITPR2','ITPR3','LDHA','LDHB','LDHC','PGAM4','ATF4','PCK1','PCK2','PDE3B','PRKAG2','PDHA1','PDHA2','PDHB','CALML5','PFKFB1','PFKL','PFKM','PFKP','PGAM1','PGAM2','PHKA1','PHKA2','PHKB','PHKG1','PHKG2','PKM','PLCB2','PLCB3','PLCB4','PRKAG3','PPARA','PPP3CA','PPP4C','PPP3CB','PPP3CC','PPP3R1','PPP3R2','PRKAA1','PRKAA2','PRKAB1','PRKAB2','PRKACA','PRKACB','PPP4R3A','PRKACG','PRKAG1','PPP4R3B','G6PC2','PYGB','PYGL','PYGM','CREB3L2','SLC2A1','SLC2A2','CALM1','CALM2','CALM3','CALML3','CAMK2A','CAMK2B','CAMK2D','CAMK2G','CREB3L3','FBP2','CREB3L1','CALML4','LDHAL6B','G6PC3','CREB5']

In [ ]:
sc.tl.score_genes(adata_hepa, gene_list=glucogen , score_name="glucogen_score")
sc.pl.umap(adata_hepa, color='glucogen_score', cmap='bwr', title='glucogen_score')

# SLC7A5 DEG analysis

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc.h5ad")
Liver_sn

In [ ]:
Liver_sn.uns['celltype_colors']

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
Liver_sn.obs['celltype'].value_counts()

In [ ]:
Liver_sn.obs['celltype'] = Liver_sn.obs['celltype'].replace({
    'Lymphocyte': 'Lymphoid',
    'Macrophage': 'Myeloid'
})

sc.pl.umap(
    Liver_sn,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False, 
    return_fig=True
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Nuc_DB_UMAP_v250112.pdf", bbox_inches='tight', dpi=300)
plt.close()

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10,
    show=False, 
    return_fig=True
)
# 직접 저장
plt.savefig("/home/jaehyunchoi/Liver_bio_project/02_SLC7A5_mono_Journal/03_Script/Nuc_DB_UMAP_v250112.png")
plt.close()

In [ ]:
Liver_sn.obs['Disease_status'].value_counts()

In [ ]:
desired_order = ["Hepatocyte", "Cholangiocyte", "Endothelial", "Stellete", "Lymphocyte", "Macrophage","B cells" ]
Liver_sn.obs['celltype'] = pd.Categorical(Liver_sn.obs['celltype'], categories=desired_order, ordered=True)

In [ ]:
Liver_stage = Liver_sn[Liver_sn.obs['Disease_status'] != 'MASLD'].copy()
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
Liver_stage

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_stage.obs['celltype']
batch_labels = Liver_stage.obs['stage']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(3, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
Liver_MASLD = Liver_stage[Liver_stage.obs['Disease_status'].isin(['Healthy', 'MASL','MASH'])].copy()
Liver_MASLD

In [ ]:
sc.pl.umap(
    Liver_MASLD,
    color='celltype',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
sc.pl.umap(
    Liver_MASLD,
    color='stage',
    frameon=False,
    title='Cell type prediction',
    size=10
)

In [ ]:
Liver_MASLD

In [ ]:
Liver_MASLD.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_MASLD.h5ad")

In [ ]:
sc.pp.neighbors(Liver_MASLD, n_pcs=30, use_rep='X_pca')

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_MASL.obs['celltype']
batch_labels = Liver_MASL.obs['stage']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(3, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_MASL.obs['celltype']
batch_labels = Liver_MASL.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
Liver_MASL.obs['stage'].value_counts()

In [ ]:
sc.tl.leiden(Liver_MASL, resolution=1, key_added="leiden")

sc.pl.umap(
    Liver_MASL,
    color='leiden',
    frameon=False,
    title='leiden annotation',
    size=10
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

cluster_labels = Liver_MASL.obs['celltype']
sample_labels = Liver_MASL.obs['sample']
stage_labels = Liver_MASL.obs['stage']   # 혹은 stage

# sample별 stage 매핑
stage_map = stage_labels.groupby(sample_labels).first()
crosstab = pd.crosstab(sample_labels, cluster_labels, normalize='index')
crosstab['stage'] = stage_map

# stage 순서 지정
stage_order = ['Healthy', 'MASL']
crosstab['stage'] = pd.Categorical(crosstab['stage'], categories=stage_order, ordered=True)

# ✅ 정렬 부분 수정
crosstab = crosstab.reset_index().sort_values(['stage', 'sample']).set_index('sample')

# 시각화
crosstab.drop(columns='stage').plot(kind='bar', stacked=True, figsize=(12, 5))
plt.title('Major cell lineages by stage (stacked bar)')
plt.ylabel('Proportion')
plt.xlabel('Samples (grouped by stage)')
plt.legend(title='Cell type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_MASL.obs['leiden']
batch_labels = Liver_MASL.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.tl.dendrogram(adata_scrna,groupby='cellannot')
sc.tl.rank_genes_groups(adata_scrna, groupby='cellannot', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(adata_scrna, n_genes=10)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(adata_scrna, group=None)
top_genes = rank_df.groupby("group")["names"].apply(lambda x: x.head(20)).explode().unique().tolist()
sc.pl.heatmap(
    adata_scrna,
    var_names=top_genes,
    groupby="cellannot",
    use_raw=False,
    cmap="bwr",
    vmin=-3,
    vmax=3,
    swap_axes=True,
    dendrogram=False,
    show=True,
    figsize=(10, 7)
)

# Liver MASLD Myeloid analysis

In [ ]:
Liver_sn_raw=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")
Liver_sn_raw

In [ ]:
Liver_sn_raw.obs['celltype']

In [ ]:
Liver_sn_raw.obs['Disease_status'].value_counts()

In [ ]:
Liver_sub= Liver_sn_raw[Liver_sn_raw.obs['Disease_status'].isin(['Healthy','MASL','MASH','Cirrhosis'])].copy()
Liver_sub

In [ ]:
Liver_Myl= Liver_sn_raw[Liver_sn_raw.obs['celltype'].isin(['Macrophage'])].copy()
Liver_Myl

In [ ]:
Liver_Myl.raw = Liver_Myl
sc.pp.normalize_total(Liver_Myl, target_sum=1e4)
sc.pp.log1p(Liver_Myl)

In [ ]:
# 변수 유전자 설정 
sc.pp.highly_variable_genes(Liver_Myl, flavor='seurat', n_top_genes=2000)
# Scale data
sc.pp.scale(Liver_Myl, max_value=10)
# Run PCA
sc.tl.pca(Liver_Myl, svd_solver='arpack')
print(Liver_Myl.obsm['X_pca'][:5, :5])
sc.pl.pca(Liver_Myl, color='sample')

import matplotlib.pyplot as plt
var_ratio = Liver_Myl.uns['pca']['variance_ratio']
plt.plot(range(1, len(var_ratio) + 1), np.cumsum(var_ratio), marker='o')
plt.xlabel('Number of Principal Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA Cumulative Variance')
plt.grid(True)
plt.show()

In [ ]:
# t-sne 
sc.tl.tsne(Liver_Myl, use_rep='X_pca', n_pcs=30)
# umap
sc.pp.neighbors(Liver_Myl, n_pcs=30, use_rep='X_pca')
sc.tl.umap(Liver_Myl)
sc.pl.umap(Liver_Myl, color=['sample'])

In [ ]:
# Harmony로 batch effect 제거
harmony_out = hm.run_harmony(Liver_Myl.obsm['X_pca'],Liver_Myl.obs, 'journal')

# Harmony 결과를 저장
Liver_Myl.obsm['X_pca_harmony'] = harmony_out.Z_corr.T

# UMAP을 사용한 시각화
sc.pp.neighbors(Liver_Myl, use_rep='X_pca_harmony')  # Harmony 결과를 사용
sc.tl.umap(Liver_Myl)
sc.pl.umap(Liver_Myl, color=['journal'])

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# 차원 축소된 데이터 (예: PCA or UMAP 등)
X = Liver_Myl.obsm['X_umap']  # 또는 'X_umap' 등

inertias = []
k_range = range(2, 20)  

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42)
    kmeans.fit(X)
    inertias.append(kmeans.inertia_)  # inertia = 클러스터 내부 거리 제곱합

# Plot
plt.figure(figsize=(6, 4))
plt.plot(k_range, inertias, marker='o')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.show()

In [ ]:
Liver_Myl

In [ ]:
# UMAP을 사용한 시각화
sc.pp.neighbors(Liver_Myl, use_rep='X_scANVI')  # Harmony 결과를 사용
sc.tl.umap(Liver_Myl)
sc.pl.umap(Liver_Myl, color=['journal'])

In [ ]:
sc.tl.leiden(Liver_Myl, resolution=0.2, key_added="leiden")
sc.pl.umap(Liver_Myl, color=['leiden'])

In [ ]:
Liver_Myl.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_monocyte.h5ad")

In [ ]:
Liver_Myl_MASL = Liver_Myl[Liver_Myl.obs['Disease_status'].isin(['Healthy', 'MASL'])].copy()
Liver_Myl_MASL

In [ ]:
Liver_Myl.obs

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl.obs['leiden']
batch_labels = Liver_Myl.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(3, 5))

# 그래프 설정
plt.title('Major cell lineages in sample bathc')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl.obs['leiden']
batch_labels = Liver_Myl.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(18, 5))

# 그래프 설정
plt.title('Major cell lineages in sample batch')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl_MASL.obs['leiden']
batch_labels = Liver_Myl_MASL.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(2, 5))

# 그래프 설정
plt.title('Major cell lineages in sample batch')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl_MASL.obs['leiden']
batch_labels = Liver_Myl_MASL.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in sample batch')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
sc.pl.umap(Liver_Myl_MASL, color=['leiden'])

In [ ]:
sc.pl.umap(Liver_Myl_MASL, color=['SLC7A5'])

In [ ]:
sc.tl.leiden(Liver_Myl_MASL, resolution=0.1, key_added="leiden")
sc.pl.umap(Liver_Myl_MASL, color=['leiden'])

In [ ]:
Liver_Myl_MASL.obs['leiden'].value_counts()

In [ ]:
Liver_Myl_MASL = Liver_Myl_MASL[Liver_Myl_MASL.obs['leiden'].isin(['0', '1','2'])].copy()
Liver_Myl_MASL

In [ ]:
Liver_Myl_MASL.write("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_monocyte_MASL.h5ad")

In [ ]:
Liver_Myl_MASL=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_monocyte_MASL.h5ad")

In [ ]:
SLCmono_genes =['PPIF','ANPEP','ITGAX','CD300E','PNP','ITGA5','FLNA','SLC11A1','ATP13A3','PHLDA1','SLC43A2','RNASEK','IFI30','SLC25A37','EZR','EEF1G','AHNAK','BACH1','GPCPD1','TOM1','ATP2B1','OLR1','NME2','PLEC','MT-ATP8','NBPF26','LIMS1','ATP6V0C','GABARAPL1','SPAG9','MAP2K3','CDKN1A','RGCC','MYADM','DENND5A','METRNL','SRRM2','SERPINB9','DUSP6','HMGA1','FMNL1','UPP1','EIF4A1','C5AR1','TREM1','ZMIZ1','MYO9B','EMILIN2','ZYX']
sc.tl.score_genes(Liver_Myl_MASL, gene_list=SLCmono_genes , score_name="SLCmono_score")
sc.pl.umap(Liver_Myl_MASL, color='SLCmono_score', cmap='bwr', title='SLCmono_score')

In [ ]:
sc.pl.tsne(Liver_Myl_MASL, color='SLCmono_score', cmap='bwr', title='SLCmono_score')

In [ ]:
sc.tl.leiden(Liver_Myl_MASL, resolution=0.5, key_added="leiden_2")
sc.pl.umap(Liver_Myl_MASL, color=['leiden_2'])

In [ ]:
sc.pl.umap(Liver_Myl_MASL, color=['leiden_2'])

In [ ]:
sc.tl.dendrogram(Liver_Myl_MASL,groupby='leiden_2')
sc.tl.rank_genes_groups(Liver_Myl_MASL, groupby='leiden_2', method='wilcoxon')
sc.pl.rank_genes_groups_dotplot(Liver_Myl_MASL, n_genes=5)

In [ ]:
rank_df = sc.get.rank_genes_groups_df(Liver_Myl_MASL, group=None)
rank_df

In [ ]:
Liver_Myl_MASL.obs['leiden_2'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(
    data=Liver_Myl_MASL.obs,
    x="leiden_2",
    y="SLCmono_score",
    palette="coolwarm"
)
plt.title("SLCmono score by cluster (leiden_2)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
gene_sets = {
    "cDC1_genes": ['XCR1', 'CLNK', 'TACSTD2', 'CLEC9A', 'EGLN3', 'IDO1', 'DNASE1L3', 'CADM1', 'S100B', 'KIAA0226L', 'WDFY4', 'CCND1', 'DUSP4', 'P2RY14', 'C1orf54', 'NRARP', 'CPNE3', 'SLAMF7', 'GPR157', 'B3GNT7', 'FLT3', 'NET1', 'RGCC', 'SERPINF1', 'BATF3', 'NDRG2', 'SNX3', 'ADAM28', 'AIM2', 'IL18R1', 'GPR171', 'PTPRCAP', 'IRF8', 'CPVL', 'C12orf45', 'TPD52', 'SLC38A1', 'LMNA', 'CD59', 'C15orf48', 'NFATC2', 'NAV1', 'MAP2K6', 'PTMS', 'ARHGAP5', 'HLA-DPB1', 'CST7', 'LGALS2', 'HLA-DQB1', 'PPA1', 'TCTN3', 'APOL1', 'CAMK2D', 'HLA-DPA1', 'CD74', 'HLA-DQA1', 'CST3', 'BCL2L11', 'DHRS3', 'ID2', 'RAB11FIP1', 'QPRT', 'OSBPL9', 'PTPN22', 'PTPN7', 'FAM129A', 'DST', 'FUCA1', 'ANPEP', 'MARCKSL1', 'HLA-DRB1', 'CLIC2', 'STK17A', 'RGS1', 'HLA-DRA', 'BASP1', 'ACTN1', 'PDLIM7', 'CD40', 'AC093673.5', 'NCOA7', 'PDE4DIP', 'TUBA1C', 'STMN1', 'CSF2RA', 'CD38', 'MIR4435-1HG', 'DUSP2', 'LGMN', 'PKIB', 'VMO1', 'SPATS2L'],
    "cDC2_genes": ['FCER1A', 'PPP1R14A', 'CD1C', 'AKR1C3', 'ENHO', 'CD1E', 'SLC38A1', 'P2RY14', 'AFF3', 'FCGR2B', 'CLEC10A', 'NDRG2', 'CST7', 'PKIB', 'CACNA2D3', 'ADAM28', 'SDPR', 'PLD4', 'NET1', 'FLT3', 'GAS6', 'ALOX5AP', 'AGPAT9', 'HLA-DQA1', 'RGS1', 'FILIP1L', 'C12orf45', 'HLA-DQB1', 'IL1R2', 'ARHGAP5', 'HLA-DPB1', 'GPR183', 'AXL', 'ADAM8', 'USP53', 'SERPINF1', 'RGCC', 'HLA-DPA1', 'CD72', 'GSN', 'CLIC2', 'ACAP1', 'ATP1B1', 'HLA-DRA', 'PMAIP1', 'CD74', 'CSF2RA', 'AREG', 'PPA1', 'FPR3', 'HLA-DRB1', 'ALCAM', 'SPATS2L', 'ARL4C', 'PTMS', 'STMN1', 'LMNA', 'VSIG4'],
    "pDC_genes":['SCT', 'LRRC26', 'LILRA4', 'MAP1A', 'PTCRA', 'LAMP5', 'CLEC4C', 'MYBL2', 'SMIM5', 'KCNK17', 'PACSIN1', 'RP11-117D22.2', 'TCL1A', 'IL3RA', 'SPIB', 'PLD4', 'BCL11A', 'RASD1', 'FAM129C', 'TPM2', 'TSPAN13', 'PTPRS', 'SERPINF1', 'PPP1R14A', 'DERL3', 'SMPD3', 'IRF4', 'ITM2C', 'PFKFB2', 'PTGDS', 'FUT7', 'SOX4', 'IRF7', 'CXCR3', 'P2RY14', 'NOTCH4', 'PMEPA1', 'LINC00996', 'PPP1R14B', 'CLN8', 'VIPR2', 'AEBP1', 'C1orf186', 'MZB1', 'IRF8', 'TCF4', 'UGCG', 'GZMB', 'TNFRSF21', 'GAPT'],
    "MigcDC_genes": ['LAD1', 'CCR7', 'LAMP3', 'CCL19', 'CCL22', 'IL7R', 'SLC7A11', 'FSCN1', 'RAMP1', 'IDO1', 'CCL17', 'CD1B', 'MIR155HG', 'TNFRSF4', 'CXCL9', 'GPR157', 'DUSP4', 'IL4I1', 'EBI3', 'TRAF1', 'MARCKSL1', 'SPIB', 'ARHGAP10', 'HLA-DQB2', 'CD1E', 'MGLL', 'IL32', 'DUSP5', 'NRP2', 'CST7', 'PSD3', 'BIRC3', 'IRF4', 'C15orf48', 'HILPDA', 'GADD45A', 'SLC38A1', 'TRAF5', 'C12orf45', 'TXN', 'RAB9A', 'TBC1D4', 'TUBA1C', 'RGS1', 'SERPINF1', 'NMRK1', 'PPA1', 'CD40', 'GSN', 'POGLUT1', 'CLIC2', 'NAV1', 'PVRL2', 'PTGIR', 'RP11-138A9.2', 'FABP5', 'ALCAM', 'FAM129A', 'NET1', 'CSF2RA', 'ANTXR2', 'UGCG', 'GPR137B', 'CD1C', 'ADAM8', 'RELB', 'PALLD', 'SLAMF7', 'ST3GAL6', 'RP11-356I2.4', 'GPR183', 'HLA-DQA1', 'NFKB1', 'FLT3', 'RP11-138A9.1', 'CERS6', 'DNASE1L3', 'P2RY8', 'CD83', 'MAFF', 'SESN1', 'CYB5A', 'PTMS', 'SDC2', 'CCL5', 'ATP1B1', 'ETV3', 'ATF5', 'HLA-DQB1', 'FILIP1L', 'PMAIP1', 'PIM3', 'ID2', 'MARCKS', 'EPSTI1', 'BHLHE40', 'SOCS1', 'BASP1', 'HLA-DPB1', 'CD74', 'APOC1', 'HLA-DRA', 'CD72', 'GBP1', 'NR4A3', 'HLA-DPA1', 'PLEK', 'LMNA', 'CLEC10A'],
    "Monocyte_genes":['S100A8', 'S100A12', 'S100A9', 'VCAN', 'CTB-61M7.2', 'THBS1', 'EREG', 'MGST1', 'RP11-1143G9.4', 'ASGR1', 'BST1', 'AQP9', 'FCN1', 'CDA', 'LYZ', 'SLC2A3', 'MEGF9', 'APOBEC3A', 'CD300E', 'CD36', 'RAB27A', 'CLEC4E', 'TMEM176A', 'SELL', 'LGALS2', 'CXCL2', 'ITGAM', 'CSF3R', 'G0S2', 'EGR1', 'SOCS3', 'CCDC69', 'CAPG', 'TREM1', 'SLC11A1', 'AREG'],
    "Patmono_genes":['LYPD2', 'CDKN1C', 'CKB', 'PTP4A3', 'CD79B', 'HES4', 'PPM1N', 'SYTL1', 'FCGR3A', 'LILRA3', 'RHOC', 'RP11-290F20.3', 'TESC', 'SPN', 'TCF7L2', 'TPPP3', 'SLC2A6', 'SLC44A2', 'RRAS', 'SNX18', 'OAS1', 'FAM110A', 'VMO1', 'MTSS1', 'LILRB2', 'PHF19', 'TBC1D10C', 'FPR2', 'LILRA5', 'IFITM2', 'SIGLEC10', 'MYO1G', 'PAG1', 'SIDT2', 'LTB', 'LST1', 'S1PR4', 'ITGAL', 'STXBP2', 'CNIH4', 'GCH1', 'IFITM3', 'KLF2'],
    "LAM_genes":['FN1', 'CST6', 'SPP1', 'MT1G', 'TREM2', 'GPNMB', 'SLC16A10', 'CD9', 'EMP1', 'APOC1', 'LMNA', 'C15orf48', 'CLEC5A', 'IL7R', 'FABP5', 'RNASE1', 'FBP1', 'SDS', 'RAB13', 'OLR1', 'APOE', 'RGCC', 'GPX3', 'TSC22D1', 'SLC39A8', 'CAPG', 'ACP5', 'MMP19', 'ALCAM', 'GSN', 'FPR3', 'PDE4DIP', 'MT1X', 'LHFPL2', 'SDC2', 'MITF', 'PLA2G7', 'LGALS3', 'PHLDA1', 'ANKRD28', 'AVPI1', 'PTMS', 'MARCO', 'B3GNT5', 'PLTP', 'GPR183', 'DAB2', 'CD59', 'RGS1', 'LGMN'],
    "MATLAM_genes":['TREM2', 'GPNMB', 'CD9', 'RNASE1', 'OTOA', 'SDS', 'A2M', 'APOC1', 'LGMN', 'FCGBP', 'FABP5', 'SLCO2B1', 'APOE', 'C1QC', 'PLA2G7', 'FPR3', 'ID3', 'DAB2', 'AXL', 'C1QB', 'PLTP', 'RGS1', 'LTC4S', 'CD72', 'GATM', 'MGLL', 'LMNA', 'ABCC5', 'GAS6', 'ANKH', 'STMN1', 'C1QA', 'ALCAM', 'MSR1', 'PKIB', 'PHLDA1', 'ATP1B1', 'GPR34', 'PTMS', 'RGCC', 'MGAT4A', 'FCGR2B', 'LIPA', 'ACP5', 'CTSL', 'MITF', 'MAF', 'EBI3', 'KCNMA1', 'MERTK', 'VSIG4', 'C15orf48', 'FILIP1L', 'SEPP1', 'SLC40A1', 'CD81', 'GPR183', 'SDC2', 'GSN', 'FOLR2', 'CD59', 'CD84', 'CPM', 'MS4A4A', 'ABCA1', 'HLA-DQA1', 'CLEC10A', 'CLIC2', 'ARL4C', 'RGL1', 'EPB41L2', 'MMP19', 'AVPI1', 'ALB', 'C1orf54', 'PDK4', 'CD1C', 'VMO1'],
    "MAC1_genes":['IGSF21', 'FCGBP', 'USP53', 'ID3', 'C1QC', 'A2M', 'SDS', 'PLTP', 'MRC1L1', 'SLCO2B1', 'LTC4S', 'CD9', 'FCGR2B', 'RGS1', 'VSIG4', 'FPR3', 'C1QB', 'GPR34', 'AXL', 'FILIP1L', 'PKIB', 'C1QA', 'CD72', 'CLEC9A', 'NDRG2', 'CLEC10A', 'GATM', 'ALB', 'GPR183', 'MSR1', 'HLA-DPA1', 'PAPOLG', 'DAB2', 'PLD4', 'LPAR6', 'PHLDA1', 'EBI3', 'F13A1', 'FCER1A', 'HLA-DQA1', 'CPM', 'HLA-DPB1', 'ALOX5AP', 'SERPINF1', 'HLA-DRB1', 'RCAN1', 'STMN1', 'HLA-DRA', 'CCL3L1', 'ATP1B1', 'HLA-DQB1', 'PRDM1', 'MAF', 'ALCAM', 'C1orf54', 'APOC1', 'PTMS', 'LGALS3BP', 'CLIC2', 'RGCC', 'SLC40A1', 'VMO1', 'SERPING1', 'EPB41L2', 'CCL3', 'FOLR2', 'CD1C', 'PDK4', 'SEPP1', 'APOE'],
    "moKC_genes":['RNASE1', 'IGSF21', 'FCGBP', 'LILRB5', 'SLCO2B1', 'C1QC', 'SEPP1', 'PLTP', 'MRC1L1', 'NRP1', 'A2M', 'C1QB', 'TMEM37', 'LYVE1', 'FRMD4A', 'GFRA2', 'APOE', 'FOLR2', 'GPR34', 'OLFML2B', 'C1QA', 'LGMN', 'MAF', 'ID3', 'CXCL12', 'AXL', 'MERTK', 'SLC40A1', 'DAB2', 'MRC1', 'DHRS3', 'PDK4', 'GATM', 'LTC4S', 'MS4A4A', 'GPNMB', 'APOC1', 'USP53', 'EBI3', 'FPR3', 'FILIP1L', 'ALB', 'STAB1', 'SDC3', 'SIGLEC1', 'KCNMA1', 'VSIG4', 'ANKH', 'PKIB', 'MSR1', 'VCAM1', 'FCHO2', 'CCL3', 'SERPING1', 'CLEC9A', 'LGALS3BP', 'CCL4', 'CCL3L1', 'CD72', 'STMN1', 'CPM', 'CD163', 'RGL1', 'ABCA1', 'CD5L', 'EPS8', 'C2', 'CLIC2', 'RGS1', 'SPRED1', 'ATP1B1', 'TSPAN4', 'RCAN1', 'LIPA', 'SMPDL3A', 'WWP1', 'FRMD4B', 'EPB41L2', 'CD81', 'TCF4', 'AP2A2', 'SPATS2L', 'CD84', 'DST', 'FMNL2', 'MS4A7', 'CD59', 'CTSL', 'PLD3', 'HLA-DPA1', 'PTMS', 'ARL4C', 'VMO1', 'C1orf54', 'EGR2', 'CLEC10A', 'FCGR3A'],
    "KC_genes":['TIMD4', 'SLC16A9', 'NDST3', 'KCNAB1', 'ITGAD', 'CD5L', 'VCAM1', 'SELENBP1', 'BCAM', 'CDH5', 'CETP', 'RND3', 'FEZ1', 'FABP3', 'SDC3', 'CXCL12', 'ITLN1', 'CTD-2337J16.1', 'LYVE1', 'SEPP1', 'TMEM37', 'MARCO', 'SCD', 'LILRB5', 'PPAP2B', 'ITGA9', 'IGF1', 'APOE', 'SLC7A8', 'EPAS1', 'SLC40A1', 'FRMD4A', 'SUCNR1', 'ACVRL1', 'IGFBP4', 'ME1', 'LGMN', 'SIGLEC1', 'SIGLEC11', 'GPR126', 'FOLR2', 'LAG3', 'ITGB5', 'NRP1', 'C1QB', 'AXL', 'MRC1', 'C1QA', 'IFI27', 'C1QC', 'C2', 'SLC46A1', 'SLCO2B1', 'ETV5', 'SCN1B', 'NR1H3', 'DHRS3', 'PDK4', 'CD209', 'CCL4L1', 'GPNMB', 'ARHGEF12', 'CD59', 'RGL1', 'ANKRD36C', 'CCL4', 'MAF', 'EPB41L2', 'GATM', 'CCND1', 'CTSF', 'IFT74', 'LIPA', 'A2M', 'MYO9A', 'ATP1B1', 'CCL4L2', 'IFIT1', 'SERPING1', 'SPATS2L', 'MERTK', 'FUCA1', 'ADORA3', 'SMPDL3A', 'CCL3', 'ABCA1', 'MRC1L1', 'ANKH', 'CMKLR1', 'DST', 'PLD3', 'PLTP', 'FSCN1', 'APOC1', 'MMD', 'QPRT', 'VSIG4', 'RHOBTB3', 'EXT1', 'DAB2', 'ALB', 'RCAN1', 'FAM213A', 'EBI3', 'GFRA2', 'CD82', 'CCL3L3', 'PLA2G15', 'WWP1', 'CD81', 'ADM', 'ABCG1', 'FXYD6', 'STOM', 'KCNMA1', 'IFIT2', 'CD38', 'PPP1R12B', 'APOC3', 'CTSL', 'LGALS3BP', 'CR1', 'MSR1', 'TNFRSF21', 'CD163', 'GCNT1', 'EPHX1', 'SASH1', 'FILIP1L', 'SNX24', 'CLIC2', 'IFIT3', 'C1orf54', 'FPR3', 'FRMD4B', 'LTC4S', 'MMP19', 'CREG1', 'HMOX1', 'ZNF189', 'SDPR', 'NCEH1', 'FMNL2', 'CCL3L1', 'ITGAV', 'TSPAN4', 'ALDH1A1', 'NFIA', 'CTSB', 'PDE4DIP', 'ARL4C', 'ARRDC3', 'STMN1', 'RBP7', 'MS4A4A', 'EGR2', 'CD84', 'TCF4', 'VMO1', 'DSC2', 'GPR34', 'FCHO2', 'SLC9A9', 'PLA2G7', 'ADAP2', 'SCARB2', 'IL10', 'FCGRT', 'PHACTR2', 'AP2A2', 'ELL2', 'ACP2', 'MS4A7', 'P4HA1', 'PDGFC', 'GPR137B', 'SGPP1', 'SPRED1', 'PKIB', 'IFI44L', 'PRNP', 'CD72', 'SIAH2', 'MS4A6A', 'EPS8', 'UGCG', 'IDH1', 'BLVRB', 'ICAM1', 'RGS1', 'CTSD', 'CD63', 'FCGR3A', 'MT1F', 'GIMAP5', 'KCNJ2']
    }
sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["Monocyte_genes"] , score_name="Monocyte_score")
sc.pl.umap(Liver_Myl_MASL, color='Monocyte_score', cmap='bwr', title='Monocyte_score')

In [ ]:
sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["MAC1_genes"] , score_name="MAC1_score")
sc.pl.umap(Liver_Myl_MASL, color='MAC1_score', cmap='bwr', title='MAC1_score')

In [ ]:
sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["LAM_genes"] , score_name="LAM_score")
sc.pl.umap(Liver_Myl_MASL, color='LAM_score', cmap='bwr', title='LAM_score')

In [ ]:
sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["Patmono_genes"] , score_name="Patmono_score")
sc.pl.umap(Liver_Myl_MASL, color='Patmono_score', cmap='bwr', title='Patmono_score')

In [ ]:
sc.tl.score_genes(Liver_Myl_MASL, gene_list=gene_sets["KC_genes"] , score_name="KC_score")
sc.pl.umap(Liver_Myl_MASL, color='KC_score', cmap='bwr', title='KC_score')

In [ ]:
myl_genes = {
    'mono':['ITGAM','CCR2','CX3CR1','S100A8'],
    'cDC': ['ITGAX','XCR1','ZBTB46','BATF3','IRF4'],
    'KC': ['TIMD4','MARCO','VSIG4']}

In [ ]:
sc.pl.dotplot(Liver_Myl_MASL, myl_genes, 
              groupby="leiden_2",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl_MASL.obs['leiden']
batch_labels = Liver_Myl_MASL.obs['sample']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(12, 5))

# 그래프 설정
plt.title('Major cell lineages in sample batch')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
# Make Stacked bargraph for the comparison of cluster compositions
cluster_labels = Liver_Myl_MASL.obs['leiden']
batch_labels = Liver_Myl_MASL.obs['Disease_status']
crosstab = pd.crosstab(batch_labels, cluster_labels, normalize='index')

import seaborn as sns
import matplotlib.pyplot as plt


# 그래프 그리기
crosstab.plot(kind='bar', stacked=True, figsize=(2, 5))

# 그래프 설정
plt.title('Major cell lineages in sample batch')
plt.ylabel('Proportion')
plt.xlabel('state')
plt.legend(title='sample proportion', bbox_to_anchor=(1.05, 1), loc='upper left')
#plt.savefig("/home/jaehyunchoi/bystander_collaboration/IPF_data/analysis/figures/3publicatlas_9cluster_state_stackedbargraph.pdf", bbox_inches='tight', dpi=300)

In [ ]:
from scipy.stats import fisher_exact
import pandas as pd

# 상태 및 클러스터 정보
state = Liver_Myl_MASL.obs['Disease_status']
cluster = Liver_Myl_MASL.obs['leiden']

# 클러스터 목록
clusters = cluster.unique()
results = []

# Donor / IPF 구분
for clus in sorted(clusters):
    # 각 조건별 개수 계산
    donor_in = ((state == "Healthy") & (cluster == clus)).sum()
    donor_out = ((state == "Healthy") & (cluster != clus)).sum()
    masl_in = ((state == "MASL") & (cluster == clus)).sum()
    masl_out = ((state == "MASL") & (cluster != clus)).sum()
    
    table = [[donor_in, donor_out], [masl_in, masl_out]]
    oddsratio, pval = fisher_exact(table)
    
    # 비율 계산
    donor_frac = donor_in / (donor_in + donor_out)
    masl_frac = masl_in / (masl_in + masl_out)
    
    results.append({
        "cluster": clus,
        "donor_fraction": donor_frac,
        "masl_fraction": masl_frac,
        "odds_ratio": oddsratio,
        "p_value": pval
    })

# 결과를 DataFrame으로 정리
df_result = pd.DataFrame(results)

# p-value 정렬
df_result = df_result.sort_values('cluster')
print(df_result)
df_result = df_result.sort_values('cluster')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(11, 6))
x = np.arange(len(df_result))
width = 0.35

# Bar plot
plt.bar(x - width/2, df_result['donor_fraction'], width, label='Healthy')
plt.bar(x + width/2, df_result['masl_fraction'], width, label='MASL')

# X축 설정
plt.xticks(x, df_result['cluster'])
plt.ylabel('Fraction')
plt.title('Myeloid cluster proportion by condition')
plt.legend()

# p-value 별로 * 표시
for i, pval in enumerate(df_result['p_value']):
    if pval < 0.001:
        stars = '***'
    elif pval < 0.01:
        stars = '**'
    elif pval < 0.05:
        stars = '*'
    else:
        stars = ''
    
    if stars:
        max_height = max(df_result['donor_fraction'][i], df_result['masl_fraction'][i])
        plt.text(x[i], 0.47, stars, ha='center', va='bottom', fontsize=12, color='black')

plt.tight_layout()
plt.show()

In [ ]:
sc.pl.umap(Liver_Myl_MASL, color=['SLC7A5'])

In [ ]:
sc.pl.umap(
    Liver_Myl_MASL,
    color='leiden',
    frameon=False,
    title='leiden',
    size=10
)

In [ ]:
sc.tl.rank_genes_groups(
    Liver_Myl_MASL,
    groupby="Disease_status",        # 비교할 obs 열
    groups=["MASL"],     # 비교할 그룹
    reference="Healthy",     # 참조 그룹
    use_raw=True,
    method="wilcoxon"       # 또는 't-test', 'logreg'
)

In [ ]:
deg_df = sc.get.rank_genes_groups_df(Liver_Myl_MASL, group="MASL")
print(deg_df.head())

In [ ]:
deg_df[deg_df['names'] == 'SLC7A5'] 

In [ ]:
filtered = deg_df[
    (deg_df['logfoldchanges'] > 0.58) &
    (deg_df['pvals_adj'] < 0.05)
]

top20_filtered = filtered.sort_values(by='scores', ascending=False).head(20)
print(top20_filtered)

In [ ]:
'SLC7A5' in filtered.index

In [ ]:
deg_df['-log10(padj)'] = -np.log10(deg_df['pvals_adj'])
genes_of_interest = ["SLC7A5","SLC3A2"]

plt.figure(figsize=(7, 6))
plt.scatter(
    deg_df['logfoldchanges'], 
    deg_df['-log10(padj)'], 
    c="grey", s=10, alpha=0.7
)

# 유의미한 cutoff 설정 (예: logFC > 1, padj < 0.05)
sig_up = (deg_df['logfoldchanges'] > 1) & (deg_df['pvals_adj'] < 0.05)
sig_down = (deg_df['logfoldchanges'] < -1) & (deg_df['pvals_adj'] < 0.05)

plt.scatter(deg_df.loc[sig_up, 'logfoldchanges'],
            deg_df.loc[sig_up, '-log10(padj)'],
            c="red", s=10, label="Upregulated")

plt.scatter(deg_df.loc[sig_down, 'logfoldchanges'],
            deg_df.loc[sig_down, '-log10(padj)'],
            c="blue", s=10, label="Downregulated")

# cutoff 선
plt.axvline(1, color="black", linestyle="--")
plt.axvline(-1, color="black", linestyle="--")
plt.axhline(-np.log10(0.05), color="black", linestyle="--")

plt.xlabel("log2 Fold Change")
plt.ylabel("-log10 Adjusted p-value")
plt.title("Volcano Plot: MASLD vs Normal")
plt.legend()

for gene in genes_of_interest:
    if gene in deg_df['names'].values:
        x = deg_df.loc[deg_df['names'] == gene, 'logfoldchanges'].values[0]
        y = deg_df.loc[deg_df['names'] == gene, '-log10(padj)'].values[0]
        plt.text(x, y, gene, fontsize=8, color="black")  # 텍스트 표시


plt.tight_layout()
plt.show()

In [ ]:
sc.pl.matrixplot(Liver_Myl_MASL, ['SLC7A5','SLC43A1'], 
              groupby="leiden_2",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(Liver_Myl_MASL, myl_genes, 
              groupby="leiden_2",
              standard_scale='var',
              figsize=(5, 3))

In [ ]:
sc.pl.dotplot(Liver_Myl_MASL, ['PPIF','ANPEP','ITGAX','CD300E','PNP','ITGA5','FLNA','SLC11A1','ATP13A3','PHLDA1'], 
              groupby="leiden_2",
              standard_scale='var',
              figsize=(5, 3))

# AGEcorrleation_bulk module6

In [ ]:
import scanpy as sc
import os

outdir = "/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Figure/"
os.makedirs(outdir, exist_ok=True)

In [ ]:
Liver_sn=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_bio_nucseq_raw.h5ad")
Liver_sn

In [ ]:
Liver_stage=sc.read("/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Data/Liver_DB_nuc_stage.h5ad")
# 순서 정의
stage_order = ["Healthy", "MASL", "MASH", "Cirrhosis", "end stage"]
Liver_stage.obs['stage'] = Liver_stage.obs['Disease_status']
Liver_stage.obs['stage'] = pd.Categorical(Liver_stage.obs['stage'], categories=stage_order, ordered=True)
Liver_stage.obs['stage_numeric'] = Liver_stage.obs['stage'].cat.codes

In [ ]:
gene = "IDI1" 
sc.pl.violin(
    Liver_stage,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1,
    save=f"Liver_sn_IDI1_stage_violinplot.png")

In [ ]:
gene = "AOX1" 
sc.pl.violin(
    Liver_stage,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1,
    save=f"Liver_sn_AOX1_stage_violinplot.png")

In [ ]:
gene = "SPP2" 
sc.pl.violin(
    Liver_stage,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1,
    save=f"Liver_sn_SPP2_stage_violinplot.png")

In [ ]:
gene = "PON1" 
sc.pl.violin(
    Liver_stage,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1,
    save=f"Liver_sn_PON1_stage_violinplot.png")

In [ ]:
gene = "DIRAS3" 
sc.pl.violin(
    Liver_stage,
    keys=gene,
    groupby="stage",
    stripplot=False, 
    jitter=0.1,
    save=f"Liver_sn_DIRAS3_stage_violinplot.png")

In [ ]:
stage_order = {
    "Healthy": 0,
    "MASL": 1,
    "MASH": 2,
    "Cirrhosis": 3,
    "end stage": 4
}

stage = Liver_stage.obs["stage"].map(stage_order)

In [ ]:
from scipy.stats import spearmanr
import numpy as np
import pandas as pd

genes = ["IDI1", "AOX1", "SPP2", "PON1", "DIRAS3"]

stages = Liver_stage.obs['stage_numeric'].values

results = []

for gene in genes:
    if gene not in Liver_stage.var_names:
        print(f"{gene} not found in Liver_stage.var_names")
        continue

    expr = Liver_stage[:, gene].X

    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()
    else:
        expr = expr.flatten()

    valid = (~np.isnan(stages)) & (~np.isnan(expr))

    rho, pval = spearmanr(stages[valid], expr[valid])

    results.append({
        "gene": gene,
        "spearman_rho": rho,
        "p_value": pval
    })

res_df = pd.DataFrame(results)
res_df

In [ ]:
import scanpy as sc

sc.pl.violin(
    Liver_stage,
    keys=["IDI1", "AOX1", "SPP2", "PON1", "DIRAS3"],
    groupby="stage",
    jitter=0.4,
    rotation=45,
    multi_panel=True
)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

genes = ["IDI1", "AOX1", "SPP2", "PON1", "DIRAS3"]
genes = [g for g in genes if g in Liver_stage.var_names]

stage = Liver_stage.obs["stage_numeric"].values

plt.figure(figsize=(6, 4))

for gene in genes:
    expr = Liver_stage[:, gene].X
    
    if hasattr(expr, "toarray"):
        expr = expr.toarray().flatten()
    else:
        expr = expr.flatten()

    df = pd.DataFrame({
        "stage": stage,
        "expression": expr
    }).dropna()

    mean_df = df.groupby("stage")["expression"].mean().reset_index()

    print(gene)
    print(mean_df)

    plt.plot(
        mean_df["stage"],
        mean_df["expression"],
        marker="o",
        label=gene
    )

plt.xlabel("Stage numeric")
plt.ylabel("Mean expression")
plt.title("Gene expression trend by stage")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"gene_expression_trend_by_stage.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
genes = ["IDI1", "AOX1", "SPP2", "PON1", "DIRAS3"]

# 1) adata.var_names에 실제 존재하는지 확인

present_genes = [g for g in genes if g in Liver_sn.var_names]
missing_genes = [g for g in genes if g not in Liver_sn.var_names]

print("존재하는 유전자:", present_genes)
print("없는 유전자:", missing_genes)

In [ ]:
sc.pl.umap(
    Liver_sn,
    color='Disease_status',
    frameon=False,
    title='Journal dataset',
    size=10
)

In [ ]:
    sc.pl.umap(
        Liver_sn,
        color='celltype',
        frameon=False,
        size=10,
        show=False,
        save=f"Liver_sn_celltype.png"
    )

In [ ]:
sc.pl.umap(
    Liver_sn,
    color=present_genes,
    frameon=False,
    size=10
)

In [ ]:
sc.pl.violin(
    Liver_sn,
    keys=present_genes,
    groupby="celltype",
    rotation=90,
    stripplot=False,
    jitter=False,
    multi_panel=True,
    use_raw = False
)

In [ ]:
sc.pl.violin(
    Liver_sn,
    keys=present_genes,
    groupby="Disease_status",
    rotation=90,
    stripplot=False,
    jitter=False,
    multi_panel=True,
    use_raw = False
)

In [ ]:
sc.pl.violin(
    Liver_sn,
    keys=present_genes,
    groupby="age",
    rotation=90,
    stripplot=False,
    jitter=False,
    multi_panel=True,
    use_raw = False
)

In [ ]:
import scanpy as sc
import os

outdir = "/home/jaehyunchoi/Liver_bio_project/50_Public_GEO_analysis/analysis/Figure/"
os.makedirs(outdir, exist_ok=True)

for gene in present_genes:
    sc.pl.umap(
        Liver_sn,
        color=gene,
        frameon=False,
        size=10,
        show=False,
        save=f"_{gene}.png"
    )

In [ ]:
# raw가 있으면 raw 사용 여부 선택
use_raw = False

expr_df = sc.get.obs_df(
    Liver_sn,
    keys=present_genes + ["celltype"],
    use_raw=use_raw
)

mean_expr = expr_df.groupby("celltype")[present_genes].mean()

mean_expr

In [ ]:
Liver_sn.obs['age'].value_counts()